# Tâche 1 - Modèles de langue N-grammes

Cette tâche consiste à construire, avec NLTK, des modèles de langue Ngrammes à partir de titres de chansons et de les utiliser pour accomplir 2 sous-tâches qui sont décrites dans l'énoncé du travail pratique. 

Les fichiers à utiliser pour cette tâche :
- *t1_titres_chansons.txt*: il contient une liste de titres de chansons, un par ligne de texte. Vous utilisez ce fichier pour la construction des modèles de langues N-grammes. 
- *t1_estimation_test.json*: il contient des listes de candidats, chaque liste contenant un seul titre de chanson connu parmi les candidats. 
- *t1_generation_test.json*: il contient des titres incomplets se terminant par des mots masqués.  

Consignes: 
- Ne pas modifier les fonctions *load_titles* et *load_tests*.
- Utilisez la variable *models* pour conserver les modèles après entraînement. 
- Ne pas modifier la signature des fonctions *train_models* et *complete_title*.
- Des modifications aux signatures de fonctions entraîneront des pénalités dans la correction. 
- Vous pouvez ajouter des cellules au *notebook* et ajouter toutes les fonctions utilitaires que vous voulez. 

## Section 1 - Lecture des fichiers de données (titres de chansons et tests)

On définit ici 2 fonctions pour lire les fichiers d'entraînement et de test. 

In [ ]:
#%pip install pandas

In [ ]:
import json


titres_fn = "./data/t1_titres_chansons.txt"    
test_estimation_fn = './data/t1_estimation_test.json'  
test_generation_fn = './data/t1_generation_test.json'  


def load_titles(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()
    return [x.strip() for x in raw_lines]


def load_tests(filename):
    with open(filename, 'r', encoding='utf-8') as fp:
        test_data = json.load(fp)
    return test_data

In [ ]:
titles = load_titles(titres_fn)

Quelques informations à propos des titres: 

In [ ]:
print("Nombre de titres pour l'entraînement: {}".format(len(titles)))

On monte également les 2 fichiers de test pour visualiser leur contenu. 

In [ ]:
import pandas as pd

def get_dataframe(tests):
    return pd.DataFrame.from_dict(tests, orient='columns', dtype=None, columns=None)

In [ ]:
tests_estimation = load_tests(test_estimation_fn)

In [ ]:
df = get_dataframe(tests_estimation)
pd.set_option("display.max_colwidth", 120)
display(df)

In [ ]:
tests_generation = load_tests(test_generation_fn)

df = get_dataframe(tests_generation)
pd.set_option("display.max_colwidth", 100)
display(df)

## Section 2 - Construction des modèles de langue N-grammes. 

La fonction ***train_models*** prend en entrée une liste de titres de chansons et construit trois modèles: unigramme, bigramme et trigramme.

Les modèles sont conservés dans ***models***, un dictionnaire Python qui prend la forme 

<pre>
{
   1: modele_unigramme, 
   2: modele_bigramme, 
   3: modele_trigramme
}
</pre>

avec comme clé la valeur N du modèle et comme valeur le modèle construit par NLTK.

In [ ]:
#pip install nltk


In [ ]:
from nltk import word_tokenize

In [ ]:
import ssl
import nltk

try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

print(nltk.download('punkt'), nltk.download('punkt_tab'))

In [ ]:
models = {} 

In [ ]:
from nltk.lm.models import Laplace

In [ ]:
from nltk.util import pad_sequence ,ngrams

BOS = '<BOS>'
EOS = '<EOS>'

In [ ]:
def build_vocabulary(text_list):
    all_unigrams = list()
    for sentence in text_list:
        word_list = word_tokenize(sentence.lower())
        all_unigrams = all_unigrams + word_list
    voc = set(all_unigrams)
    voc.add(BOS)
    voc.add(EOS)
    return list(voc)

In [ ]:
def get_ngrams(text_list, n=2):
    all_ngrams = list()
    for sentence in text_list:
        tokens = word_tokenize(sentence.lower())
        padded_sent = list(pad_sequence(tokens, pad_left=True, left_pad_symbol=BOS,
                                        pad_right=True, right_pad_symbol=EOS, n=n))
        all_ngrams = all_ngrams + list(ngrams(padded_sent, n=n))
    return all_ngrams

In [ ]:
def train_models(texts):
    """ Vous ajoutez à partir d'ici le code dont vous avez besoin
        pour construire les différents modèles N-grammes.
        Cette fonction doit construire tous les modèles en une seule passe.
    """   
    vocabulary = build_vocabulary(texts)
    for n in range(1,4):
        corpus_ngrams = get_ngrams(texts, n=n)
        model = Laplace(n)
        model.fit([corpus_ngrams], vocabulary_text=vocabulary)
        models[n] = model


In [ ]:
train_models(titles)
print(models)

In [ ]:
print(len(models[1].vocab))
print("love" in models[1].vocab)

TEST SELECTION CANDUDAT

In [ ]:
candidat = ["let it be"]
print(get_ngrams(candidat, n=2))

In [ ]:
ng_be   = get_ngrams(["let it be"], n=2)
ng_roll = get_ngrams(["let it roll"], n=2)
print(models[2].perplexity(ng_be))
print(models[2].perplexity(ng_roll))


In [ ]:
candidates = ["let it be", "let me be", "let it roll"]
n = 2

best_score = float("inf")      # l'infini : n'importe quel score sera plus petit au départ
result = None

for candidate in candidates:
    ngrams_cand = get_ngrams([candidate], n=n)
    score = models[n].perplexity(ngrams_cand)
    if score < best_score:
        best_score = score
        result = candidate

print(result, best_score)

## Section 3 - Estimation - Choix parmi plusieurs candidats

In [ ]:
def select_candidate(candidates, n=3):
    """ 
        Cette fonction retourne le candidat retenu ainsi que son score attribué par le modèle.
        
        *candidates* est la liste de passages de textes obtenu à partir du fichier de test. 
        Un seul des passage est un titre de chanson connu. 
        
        Le paramètre n désigne le modèle utilisé.
        1 - unigramme NLTK, 2 - bigramme NLTK, 3 - trigramme NLTK
         
    """

    best_score = float("inf")
    result = None
    for candidate in candidates:
        ngrams_cand = get_ngrams([candidate], n=n)
        score = models[n].perplexity(ngrams_cand)
        if score < best_score:
            best_score = score
            result = candidate
    return result, best_score


In [ ]:
print(select_candidate(["i want to hold and love","i want to hold something", "i want to hold your hand"], n=2))

In [ ]:
for n in range(1, 4):
    correct = 0
    for test in tests_estimation:
        choix, score = select_candidate(test["Candidates"], n=n)
        if choix == test["Song"]:
            correct = correct + 1
    print("Modèle", n, ":", correct, "/", len(tests_estimation))

Présentez ici vos résultats et répondez aux questions formulées dans l'énoncé du travail. 

## Section 4 - Génération - Compléter les titres de chansons

In [ ]:
import random
from collections import Counter
from nltk import word_tokenize

# Ensemble de symbols de padding lors de la tokinization
PADDING_SYMBOLES = [('<BOS>', '<EOS>'), ('<s>', '</s>')]

# Retrouve les symboles de début et de fin utilisés à l'entraînement du modèle.
def get_pad_symbols(model):
    for bos, eos in PADDING_SYMBOLES:
        if bos in model.vocab:
            return bos, eos
    return None, None
    
def fallback_distribution(model):
    """ Fréquences des mots en position de prédiction, tous contextes confondus
        (utilisé quand aucun contexte connu n'est disponible). """
    freq = Counter(model.counts.unigrams)
    if not freq:  # modèle entraîné uniquement avec des N-grammes d'ordre N
        cfd = model.counts[model.order]
        for ctx in cfd.conditions():
            freq.update(cfd[ctx])
    for tok in special_tokens(model):
        freq.pop(tok, None)
    return freq

In [ ]:
# --- TEMPORAIRE : modèles de test en attendant train_models (section 2). À retirer avant la remise. ---
from nltk.util import pad_sequence, ngrams
from nltk.lm.models import Laplace

def _train_tmp(texts):
    tmp = {}
    vocab = {'<BOS>', '<EOS>'}
    grams = {1: [], 2: [], 3: []}
    for t in texts:
        tokens = word_tokenize(t.lower())
        vocab.update(tokens)
        for n in (1, 2, 3):
            padded = list(pad_sequence(tokens, pad_left=True, left_pad_symbol='<BOS>', pad_right=True, right_pad_symbol='<EOS>', n=n))
            grams[n].extend(ngrams(padded, n=n))
    for n in (1, 2, 3):
        tmp[n] = Laplace(n)
        tmp[n].fit([grams[n]], vocabulary_text=list(vocab))
    return tmp

tmp_models = _train_tmp(titles)
print({n: tmp_models[n].order for n in tmp_models})   # {1: 1, 2: 2, 3: 3}

In [ ]:
def sample_next_word(model, context, rng, max_tries=50):
    """ Tire le mot suivant avec model.generate (NLTK). Si le contexte est inconnu ou ne mène
        qu'à des jetons spéciaux, on se replie sur un contexte plus court. """
    specials = special_tokens(model)
    history = list(context[-(model.order - 1):]) if model.order > 1 else []
    while True:
        for _ in range(max_tries):
            try:
                word = model.generate(1, text_seed=history, random_seed=rng)
            except ValueError:          # aucun N-gramme connu pour ce contexte
                break
            if word not in specials:
                return word
        if not history:
            break
        history = history[1:]           # repli : on retire le mot le plus ancien du contexte
    # Dernier recours : tirage selon les fréquences globales des mots
    freq = fallback_distribution(model)
    words = sorted(freq)
    return rng.choices(words, weights=[freq[w] for w in words])[0]


In [ ]:
def complete_title(model, tokenized_seed, nb_words, random_seed=46):
    """
        Génère des mots et les ajoute à la fin du texte tokenisé. 
        Retourne la liste de mots du titre complet (c.-à-d. le *tokenized_seed* + les *nb_words* générés).
        
        model: Le modèle N-gramme utilisé
        tokenized_seed: le titre incomplet, sans les masques, qui a été tokenisé. Par exemple: ["le", "soleil"].
        random_seed: utilisé par NLTK pour la génération de nombre aléatoire. 
        
    """    

    # Votre code à partir d'ici. Vous pouvez modifier comme bon vous semble. Retirez cette ligne. 
    
    rng = random.Random(random_seed)    # un seul générateur : tirages différents à chaque mot, mais reproductibles
    context = padded_context(model, tokenized_seed)
    new_words = []
    for _ in range(nb_words):
        word = sample_next_word(model, context, rng)
        new_words.append(word)
        context.append(word)  # le mot généré devient le contexte du suivant

    generated = list(tokenized_seed) + new_words
    return generated

# des prints pour tester notre modèle
print(complete_title(tmp_models[3], ["i", "want", "to"], 2))
print(complete_title(tmp_models[3], ["i", "believe", "i"], 2))
print(complete_title(tmp_models[3], ["mot", "inconnu"], 2))   # contexte jamais vu

## Section 5 - Partie réservée pour faire nos tests lors de la correction

Merci de ne pas modifier ni retirer cette section du notebook.  